# 3.3 Train residual stackers

**Inputs:** validation meta-features and actual prices.

**Outputs:** 96 residual stackers and one validation price prediction per horizon.

`predicted_price = base_l1_prediction + predicted_residual`

In [1]:
# Step 1 - Imports and stacker settings

import os
import sys

import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import psutil
import pyarrow.parquet as pq
from tqdm import tqdm

sys.path.append(os.path.abspath("../../")) ; from EPF import variables

HORIZONS = range(1, variables.HORIZON_COUNT+1)
saved_columns = pq.ParquetFile(variables.VALIDATION_META_FEATURES_PATH).schema_arrow.names
META_FEATURES = [column.removesuffix("_h1") for column in saved_columns if column.endswith("_h1")]
STACKER_PARAMS = {
    "objective": "regression_l1", "metric": "mae", "n_estimators": 800,
    "learning_rate": 0.03, "num_leaves": 15, "max_depth": 4,
    "min_child_samples": 500, "subsample": 0.8, "subsample_freq": 1,
    "colsample_bytree": 0.9, "reg_alpha": 0.1, "reg_lambda": 5.0,
    "random_state": 1729, "force_col_wise": True, "verbosity": -1,
    "n_jobs": max(1, psutil.cpu_count(logical=False)-2),
}

In [2]:
# Step 2 - Load actual validation prices

def read_period(path, columns):
    index_col = (pq.ParquetFile(path).schema_arrow.pandas_metadata or {})["index_columns"][0]
    return pd.read_parquet(path, columns=columns, filters=[(index_col, ">", variables.VALID_START), (index_col, "<=", variables.TEST_START)]).sort_index().astype(np.float32)

target_columns = [f"target_h{horizon}" for horizon in HORIZONS]
validation_targets = read_period(variables.AGG_TARGET_DATASET_PATH, target_columns)
validation_predictions = pd.DataFrame(index=validation_targets.index)
diagnostics = []

variables.RESIDUAL_STACKERS_DIR.mkdir(parents=True, exist_ok=True)

In [3]:
# Step 3 - Train and save one stacker per horizon

for horizon in tqdm(HORIZONS):
    columns = [f"{name}_h{horizon}" for name in META_FEATURES]
    meta = pd.read_parquet(variables.VALIDATION_META_FEATURES_PATH, columns=columns).reindex(validation_targets.index)
    meta.columns = META_FEATURES
    actual = validation_targets[f"target_h{horizon}"]
    lead_time = pd.Timedelta(minutes=horizon*variables.HORIZON_GRANULARITY_IN_MINUTES)
    keep = (validation_targets.index+lead_time <= variables.TEST_START) & actual.notna() & meta.notna().all(axis=1)
    meta = meta.loc[keep]
    actual = actual.loc[keep]
    residual = actual-meta["base_l1"]

    tune_start = round(len(actual)*0.8)
    purge_rows = int(np.ceil(horizon*variables.HORIZON_GRANULARITY_IN_MINUTES/variables.FEATURE_GRANULARITY_IN_MINUTES))
    train_stop = tune_start-purge_rows

    trial = lgb.LGBMRegressor(**STACKER_PARAMS)
    trial.fit(meta.iloc[:train_stop], residual.iloc[:train_stop], eval_X=meta.iloc[tune_start:], eval_y=residual.iloc[tune_start:], callbacks=[lgb.early_stopping(60, verbose=False)])
    best_iteration = int(trial.best_iteration_)
    tune_prediction = meta["base_l1"].iloc[tune_start:] + trial.predict(meta.iloc[tune_start:], num_iteration=best_iteration)

    stacker = lgb.LGBMRegressor(**{**STACKER_PARAMS, "n_estimators": best_iteration}).fit(meta, residual)
    joblib.dump(stacker, variables.RESIDUAL_STACKERS_DIR / f"h{horizon:02d}_residual_stacker.joblib")
    predicted_price = meta["base_l1"] + stacker.predict(meta)
    validation_predictions[f"predicted_h{horizon}"] = predicted_price

    tune_actual = actual.iloc[tune_start:]
    diagnostics.append({
        "horizon": horizon,
        "best_iteration": best_iteration,
        "base_validation_mae": float(np.abs(meta["base_l1"].iloc[tune_start:]-tune_actual).mean()),
        "stacked_validation_mae": float(np.abs(tune_prediction-tune_actual).mean()),
    })

validation_predictions.index.name = "date"
validation_predictions.to_parquet(variables.VALIDATION_CENTRAL_PREDICTIONS_PATH)
diagnostics = pd.DataFrame(diagnostics)
diagnostics.to_csv(variables.RESIDUAL_STACKER_DIAGNOSTICS_PATH, index=False)

display(diagnostics.head(12))
print(variables.RESIDUAL_STACKERS_DIR)
print(variables.VALIDATION_CENTRAL_PREDICTIONS_PATH)

100%|██████████| 96/96 [02:31<00:00,  1.58s/it]


,horizon,best_iteration,base_validation_mae,stacked_validation_mae
0,1,593,77.759605,76.743823
1,2,121,84.223259,83.492544
2,3,131,87.273643,86.347811
3,4,207,89.088753,88.121030
4,5,170,89.403854,88.990922
5,6,32,89.803085,89.498995
6,7,126,91.492126,89.611642
7,8,110,91.057892,89.138495
8,9,167,91.943497,90.039588
9,10,64,91.625778,90.239834


/home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/4_combine_models/3_residual_stackers
/home/daniel-davaris/Documents/NEM-Short-Term-Price-Forecasting/EPF/5_Model/Data/4_combine_models/3_validation_predictions.parquet
